In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

import os
import shutil
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=12,
    executor_memory="32g",
    executor_cores=4,
    update_configs={
        "spark.sql.shuffle.partitions": "512",  # 6.2B ÷ 500 = 12.4M rows/partition
    },
    aws_profile="admin-user"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
base_dir = "/data/data_processing/backfill-nwm/nwm-analysis-assim-streamflow-teehr-warehouse/local/cache/fetching/nwm"

# nwm_configuration = "nwm30_analysis_assim_no_da" # loaded
# nwm_configuration = "nwm30_analysis_assim_extend_no_da" # loaded
# nwm_configuration = "nwm30_analysis_assim_alaska_no_da"  # loaded
# nwm_configuration = "nwm30_analysis_assim_extend_alaska_no_da" # loaded
# nwm_configuration = "nwm30_analysis_assim_hawaii_no_da" # loaded
# nwm_configuration = "nwm30_analysis_assim_puertorico_no_da" # loaded

variable_name = "streamflow_hourly_inst"

cache_dir = Path(base_dir, nwm_configuration, variable_name).as_posix()
cache_dir

In [ ]:
files = list(Path(cache_dir).glob("**/*.parquet"))
total_size = sum(f.stat().st_size for f in files)
print(f"Files: {len(files)}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
nwm_sdf = ev.spark.read.parquet(cache_dir)
nwm_sdf.count()

In [ ]:
nwm_sdf.selectExpr("min(value_time)").show(), nwm_sdf.selectExpr("max(value_time)").show()

In [ ]:
%%time
ev.secondary_timeseries.load_dataframe(nwm_sdf)

In [ ]:
sdf = ev.secondary_timeseries.filter([
    f"configuration_name = '{nwm_configuration}'"
]).to_sdf()

In [ ]:
sdf.count()

In [ ]:
sdf.selectExpr("min(value_time)").show()

In [ ]:
spark.stop()

#### NOTE: If you are confident the data has been loaded to the warehouse delete the local evaluation data cached at: 

In [ ]:
cache_dir